# EDU Ring-Down: Pre-Vibe vs Post-Vibe Analysis and Q-Estimation Investigation

This notebook has two goals:

1. **Part I (sections 1–4):** Compare the EDU R1 resonator ring-down measurements taken before
   and after the vibration test — frequencies, quality factors, and data-quality anomalies.
2. **Part II (sections 5–6):** Investigate *why* the library's Q estimators
   (`Q_profile`, `Q_nls`, `Q_envelope`) disagree with each other and with the visually
   obvious decay envelope on these real records, using diagnostics and controlled
   synthetic experiments that isolate one effect at a time.

**Data** (Moku:Pro Phasemeter exports, acquisition rate ≈ 149.01 Hz):

| Epoch | File | Ring-down channel |
|---|---|---|
| Pre-Vibe (2026-03-21) | `data/ODIN/20260321_EDU_R1.csv.zip` | ch1 = PD1M (ch2 = PD1R) |
| Post-Vibe (2026-03-25) | `data/ODIN/20260325_EDU_R1.csv.zip` | ch2 = PD1R (ch1 = PD1M, ch3 = PD2M, ch4 = PD2R) |

Channel-to-photodiode mapping comes from the CSV header comments. Both files begin
essentially at the release epoch (the ring-down is already decaying at the first sample).

**How to run:** activate the project venv (`source .venv/bin/activate`) and run all cells
from the repo's `notebooks/` directory. Requires `mokutools` (installed in the venv) for
loading the phasemeter zips. Full execution takes ≈ 10–15 minutes; the analyzer runs and
the segmented demodulation dominate the runtime. All random seeds are fixed.

Key figures are also written to `docs/investigations/figures/` for the accompanying
investigation report `docs/investigations/20260818_q_estimation_failure_investigation.md`.

> **Historical note (2026-08-19).** This notebook is the *evidence record* for the
> Q-estimation failures: it was executed against the library as of commit `f20ef12`,
> **before** the fixes it motivated were implemented. The library-evolution plan proposed
> here has since landed: the `tau_est` crop cascade is guarded, `Q_profile` is gated on
> envelope mismatch and on frequency drift, and the segmented demodulation of section 3
> now ships as `ringdownanalysis.demod.SegmentedDemodEstimator` (with `Q_demod` /
> `Q_selected` pipeline fields, a nonlinear-damping model in `ringdownanalysis.nonlinear`,
> and estimator selection in `ringdownanalysis.selection`). **Re-running the analyzer
> cells against the current library will therefore produce different — now guarded —
> outputs**; the stored outputs document the historical behavior and should not be
> regenerated. For the current-library behavior on these same pathologies, see
> `notebooks/20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb`; the real-data numbers in
> sections 3–4 are pinned by `tests/test_real_data_regression.py`.

In [ ]:
import logging
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import welch

from mokutools.phasemeter import MokuPhasemeterObject
from ringdownanalysis import RingDownAnalyzer, configure_logging, plots
from ringdownanalysis.plots import plot_q_envelope_overlay
from ringdownanalysis.q_envelope import q_envelope_diagnostic
from ringdownanalysis.q_profile import ProfileQEstimator

configure_logging(level=logging.ERROR)
plots.apply_plotting_style()
plt.style.use("default")
%matplotlib inline

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "ODIN"
FIG_DIR = REPO_ROOT / "docs" / "investigations" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PRE_FILE = DATA_DIR / "20260321_EDU_R1.csv.zip"
POST_FILE = DATA_DIR / "20260325_EDU_R1.csv.zip"
for f in (PRE_FILE, POST_FILE):
    assert f.exists(), f"Missing data file: {f}"

# Ring-down channels established in section 1
PRE_CH, POST_CH = 1, 2
F_NOMINAL = 7.6699  # Hz, approximate R1 resonance (refined below)

RNG_SEED = 20260818
analyzer = RingDownAnalyzer()

def savefig(fig, name):
    fig.savefig(FIG_DIR / name, dpi=130, bbox_inches="tight")

print("Setup OK. Figures ->", FIG_DIR)

## 1. Data loading and channel overview

Load the complete records once and keep them in memory. Note a `mokutools` convention that
matters for reproducibility: `MokuPhasemeterObject(start_time=...)` is an **offset from the
first sample of the file**, not an absolute value of the time column. The Pre-Vibe file's
time column starts at ≈ 450 s and the Post-Vibe file's at ≈ 9000 s; in both cases
`start_time=0` starts right at the release. (Passing `start_time=9000` to the Post-Vibe
file — as a reading of the older Day-05 notebooks might suggest — actually selects data
starting 9000 s *after* the release; section 5.1 uses that window deliberately as a
failure specimen.)

In [ ]:
# Load full records (time axes re-zeroed to the first sample)
records = {}
for label, (path, nchan) in {"Pre-Vibe": (PRE_FILE, 2), "Post-Vibe": (POST_FILE, 4)}.items():
    pm = MokuPhasemeterObject(filename=str(path), start_time=0, duration=3600 * 24)
    t = pm.df["time"].values
    records[label] = {
        "t": t - t[0],
        "t_abs0": t[0],
        "fs": pm.fs,
        "channels": {ch: pm.df[f"{ch}_cycles"].values for ch in range(1, nchan + 1)},
    }
    print(
        f"{label}: fs = {pm.fs:.3f} Hz, duration = {pm.duration:.0f} s, "
        f"nchan = {nchan}, time column starts at {t[0]:.1f} s"
    )

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
for ax, (label, rec) in zip(axes, records.items()):
    for ch, y in rec["channels"].items():
        ax.plot(rec["t"][::50], y[::50], lw=0.6, alpha=0.85, label=f"ch{ch}")
    ax.set_title(f"{label}: all channels (time relative to file start)")
    ax.set_xlabel("Time since file start (s)")
    ax.set_ylabel("Phase (cycles)")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)
plt.tight_layout()
savefig(fig, "fig01_overview.png")
plt.show()

### 1.1 Which channels actually rang down?

For each channel, compare the power of the ≈ 7.67 Hz resonance tone against the broadband
background in the first ≈ 2 h. A genuine ring-down stands many orders of magnitude above
background; weak crosstalk pickup from the ringing resonator shows up at only $10^2$–$10^3$.

In [ ]:
rows = []
for label, rec in records.items():
    m = rec["t"] < 8000.0
    for ch, y in rec["channels"].items():
        yd = y[m]
        x = np.arange(len(yd))
        yd = yd - np.polyval(np.polyfit(x, yd, 2), x)  # remove slow drift
        f, p = welch(yd, fs=rec["fs"], nperseg=2**16)
        band = (f > 7.0) & (f < 8.4)
        bg = (f > 1.0) & (f < 60.0) & ~band
        f_peak = f[band][np.argmax(p[band])]
        ratio = p[band].max() / np.median(p[bg])
        rows.append(
            {
                "epoch": label,
                "channel": f"ch{ch}",
                "f_peak_Hz": round(float(f_peak), 4),
                "tone_to_background": f"{ratio:.1e}",
                "phase_ptp_cycles": f"{np.ptp(y[m]):.1e}",
                "verdict": "RING-DOWN" if ratio > 1e8 else "crosstalk only",
            }
        )
tone_df = pd.DataFrame(rows)
tone_df

**Verdict:** the R1 resonator rang down on both days — Pre-Vibe on **ch1 (PD1M)** and
Post-Vibe on **ch2 (PD1R)** — with the tone $\sim 10^{13}$ above background at the same
frequency (≈ 7.669 Hz). All other channels only see weak ($10^2$–$10^3$) crosstalk pickup
of the same tone; there is no independent R2 ring-down in either file. The remainder of the
notebook therefore compares **R1 Pre-Vibe vs Post-Vibe**.

Two data-quality observations that will matter later (visible in the overview plot):

- The oscillation **never decays to zero**. After ≈ 10 000–15 000 s both records settle
  into a fluctuating band of ≈ ±15–50 cycles with deep nulls and revivals
  (Post-Vibe shows a large revival near +48 000 s). Section 3 shows this is the resonator
  itself, persistently driven by ambient excitation — not instrument noise.
- The baseline (DC phase) wanders by ≈ 10–25 cycles over hours in both records.

## 2. Baseline: library analyzer on the canonical windows

Run `RingDownAnalyzer.analyze_array` on the analysis windows used by the earlier EDU
notebooks (Pre-Vibe: first 3 h with `tau_init=2000`; Post-Vibe: first 3.5 h with
`detrend="constant"`), then draw `plot_q_envelope_overlay` for each. This is the
baseline that exposes the Q-estimation problem investigated in Part II.

In [ ]:
def window(label, ch, start, duration):
    rec = records[label]
    m = (rec["t"] >= start) & (rec["t"] <= start + duration)
    return rec["t"][m] - rec["t"][m][0], rec["channels"][ch][m]

canonical = {
    "Pre-Vibe": dict(ch=PRE_CH, start=0.0, duration=3600 * 3.0, kwargs={"tau_init": 2000.0}),
    "Post-Vibe": dict(ch=POST_CH, start=0.0, duration=3600 * 3.5, kwargs={"detrend": "constant"}),
}

canon_results = {}
canon_rows = []
for label, cfg in canonical.items():
    t_w, y_w = window(label, cfg["ch"], cfg["start"], cfg["duration"])
    t0 = time.time()
    r = analyzer.analyze_array(t=t_w, data=y_w, **cfg["kwargs"])
    canon_results[label] = r
    canon_rows.append(
        {
            "epoch": label,
            "f_nls_Hz": r["f_nls"],
            "f_dft_Hz": r["f_dft"],
            "tau_est_s": r["tau_est"],
            "Q_profile": r["Q_profile"],
            "Q_profile_status": r["Q_profile_status"],
            "Q_profile_ci95": r["Q_profile_ci95"],
            "Q_nls_raw": r["Q_nls_raw"],
            "Q_nls_status": r["Q_nls_status"],
            "Q_envelope": r["Q_envelope"],
            "tau_envelope_s": r["tau_envelope"],
            "envelope_agrees_with_best_Q": r["Q_envelope_candidate_agrees"],
            "runtime_s": round(time.time() - t0, 1),
        }
    )
canon_df = pd.DataFrame(canon_rows).set_index("epoch")
canon_df.T

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9))
for ax, (label, r) in zip(axes, canon_results.items()):
    plot_q_envelope_overlay(ax, r)
    ax.set_title(f"{label}: plot_q_envelope_overlay (q_source='best')")
plt.tight_layout()
savefig(fig, "fig02_overlay_canonical.png")
plt.show()

**The symptom, reproduced.** On both records:

- The red *"profile Q"* envelope (the estimator the library recommends and the batch API
  prefers) decays **visibly faster than the data** — on Pre-Vibe roughly 1.5× too fast, on
  Post-Vibe roughly 4× too fast.
- Yet `Q_profile_status` is **`valid`** with an absurdly tight 95 % confidence interval
  (relative width $\sim 10^{-4}$). The interval is statistically meaningless here — a first
  hint that the fit's noise model does not describe these data.
- The three estimators disagree wildly with each other
  (Pre-Vibe: `Q_profile` ≈ 5.8×10⁴, `Q_nls_raw` ≈ 1.2×10⁵, `Q_envelope` ≈ 9.4×10⁴;
  Post-Vibe: ≈ 2.5×10⁴ / 9.1×10⁴ / 1.1×10⁵). `Q_profile` is the only one claiming
  `valid`; the envelope diagnostic flags the mismatch (`candidate_q_envelope_mismatch`).
- `tau_est` (the full-record coherent fit that also sets the crop) lands at 17 692 s on
  Pre-Vibe — 4.6× the envelope τ of 3888 s — and at 1475 s on Post-Vibe — 0.33× of
  4474 s. The coherent fit has no stable optimum on these data: earlier sessions running
  the *same* Pre-Vibe window (differing only at numerical round-off level) returned
  `Q_profile` ≈ 4.0×10⁴ instead of the 5.8×10⁴ seen here. Section 5.1 demonstrates this
  ill-conditioning directly.

Which one (if any) is right? Part I first builds a **drift-immune** measurement of the
decay that we can trust, then uses it for the Pre/Post comparison; Part II explains the
failures mechanistically.

> *Historical behavior.* Under the current library these same windows no longer return a
> confidently-wrong `valid` `Q_profile`: the drift gate (fired by the demod estimator's
> coherence diagnostic) and the envelope-mismatch gate demote the coherent estimates, and
> `Q_demod` / `Q_selected` carry the trustworthy value.

## 3. Drift-immune diagnostics: segmented demodulation

The library's NLS and profile-likelihood estimators fit a **globally phase-coherent**
model $A_0 e^{-t/\tau}\cos(2\pi f t + \phi) + c$: a single constant frequency and phase
over the whole (multi-hour) window. To measure what the signal actually does without that
assumption, we demodulate in short segments:

- Split the record into 120 s segments (≈ 920 oscillation cycles each — long enough for
  a precise per-segment fit, short enough that frequency drift within a segment is
  negligible).
- Per segment: remove mean and linear baseline drift, locate the tone with a zero-padded
  FFT, then refine with a two-stage scan of linear least-squares fits
  $a\cos(2\pi f t)+b\sin(2\pi f t)+c$ over $f$.
- Report the segment's amplitude $A=\sqrt{a^2+b^2}$, frequency $f$, and residual RMS.

This yields amplitude and frequency **as functions of time**, immune to phase drift and
to the choice of the global analysis window. It is used both as a diagnostic and, in
section 4, as the reference ("ground-truth") estimator for the Pre/Post comparison.

In [ ]:
def demodulate_segments(t, y, fs, seg_dur=120.0, f_band=(7.5, 7.85)):
    """Per-segment amplitude/frequency by FFT seed + two-stage fixed-f linear LSQ scan.

    Returns array with columns [t_mid (s, relative), f (Hz), A, sigma_resid].
    """
    out = []
    n_seg = int((t[-1] - t[0]) / seg_dur)
    for i in range(n_seg):
        s = t[0] + i * seg_dur
        lo = np.searchsorted(t, s)
        hi = np.searchsorted(t, s + seg_dur)
        if hi - lo < 100:
            continue
        ts = t[lo:hi] - t[lo]
        ys = y[lo:hi] - np.mean(y[lo:hi])
        ys = ys - np.polyval(np.polyfit(ts, ys, 1), ts)  # per-segment baseline
        spec = np.abs(np.fft.rfft(ys * np.hanning(len(ys)), n=4 * len(ys)))
        fr = np.fft.rfftfreq(4 * len(ys), 1.0 / fs)
        band = (fr > f_band[0]) & (fr < f_band[1])
        f_seed = fr[band][np.argmax(spec[band])]

        def scan(f_center, half_span, n_pts):
            best = None
            for f in f_center + np.linspace(-half_span, half_span, n_pts):
                design = np.column_stack(
                    [np.cos(2 * np.pi * f * ts), np.sin(2 * np.pi * f * ts), np.ones_like(ts)]
                )
                coef, _, _, _ = np.linalg.lstsq(design, ys, rcond=None)
                rss = float(np.sum((ys - design @ coef) ** 2))
                if best is None or rss < best[0]:
                    best = (rss, f, float(np.hypot(coef[0], coef[1])))
            return best

        rss, f_c, _ = scan(f_seed, 2e-3, 21)
        rss, f_hat, a_hat = scan(f_c, 2e-4, 21)
        sigma = np.sqrt(rss / (len(ys) - 3))
        out.append((s + seg_dur / 2 - t[0], f_hat, a_hat, sigma))
    return np.array(out)


def decay_region_fit(t_mid, amp, freq, floor):
    """Floor-corrected log-linear fit over the contiguous early region with A > 3*floor.

    Returns (mask, tau, Q, slope_stderr)."""
    above = amp > 3.0 * floor
    idx = np.flatnonzero(~above)
    t_end = t_mid[idx[0]] if len(idx) else t_mid[-1]
    m = above & (t_mid < t_end)
    amp_c = np.sqrt(np.maximum(amp**2 - floor**2, 1e-12))  # subtract driven floor in power
    slope, intercept = np.polyfit(t_mid[m], np.log(amp_c[m]), 1)
    resid = np.log(amp_c[m]) - (slope * t_mid[m] + intercept)
    sxx = np.sum((t_mid[m] - np.mean(t_mid[m])) ** 2)
    slope_err = np.sqrt(np.sum(resid**2) / (np.sum(m) - 2) / sxx)
    tau = -1.0 / slope
    Q = np.pi * np.mean(freq[m]) * tau
    return m, tau, Q, slope_err


demod = {}
for label, ch in {"Pre-Vibe": PRE_CH, "Post-Vibe": POST_CH}.items():
    rec = records[label]
    t0 = time.time()
    seg = demodulate_segments(rec["t"], rec["channels"][ch], rec["fs"])
    demod[label] = seg
    print(f"{label}: {len(seg)} segments demodulated in {time.time() - t0:.0f} s")

In [ ]:
gt = {}  # ground-truth decay parameters per epoch
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for j, (label, seg) in enumerate(demod.items()):
    t_mid, f_seg, a_seg, sig_seg = seg.T
    floor = float(np.median(a_seg[t_mid > 0.75 * t_mid[-1]]))
    m, tau_gt, q_gt, slope_err = decay_region_fit(t_mid, a_seg, f_seg, floor)
    gt[label] = dict(t_mid=t_mid, f=f_seg, A=a_seg, sigma=sig_seg, floor=floor,
                     decay_mask=m, tau=tau_gt, Q=q_gt, slope_err=slope_err)

    ax = axes[0, j]
    ax.semilogy(t_mid, a_seg, ".", ms=3, label="demodulated amplitude")
    ax.axhline(floor, color="r", ls=":", label=f"driven plateau ≈ {floor:.1f} cycles")
    tt = t_mid[m]
    ax.semilogy(tt, a_seg[m][0] * np.exp(-(tt - tt[0]) / tau_gt), "g-", lw=2,
                label=f"fit: tau = {tau_gt:.0f} s, Q = {q_gt:.3g}")
    ax.set_title(f"{label}: amplitude vs time")
    ax.set_xlabel("Time since release (s)")
    ax.set_ylabel("Amplitude (cycles)")
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[1, j]
    ax.plot(t_mid[m], f_seg[m], ".", ms=4)
    ax.set_title(f"{label}: frequency vs time (decay region)")
    ax.set_xlabel("Time since release (s)")
    ax.set_ylabel("f (Hz)")
    ax.grid(alpha=0.3)
    print(
        f"{label}: plateau = {floor:.1f} cycles | decay region 0..{t_mid[m][-1]:.0f} s "
        f"({np.sum(m)} segments) | tau = {tau_gt:.0f} s -> Q = {q_gt:.4g} | "
        f"f drift over decay = {(f_seg[m][-1] - f_seg[m][0]) * 1e3:+.3f} mHz"
    )
plt.tight_layout()
savefig(fig, "fig03_demod_amplitude_frequency.png")
plt.show()

**Three properties of the real signal, none of which the library's model includes:**

1. **The frequency is not constant.** It rises smoothly by ≈ +1.1 mHz (Pre-Vibe) /
   ≈ +0.3 mHz (Post-Vibe) *during* the decay — an amplitude-dependent (anharmonic)
   frequency pull, examined in section 3.2. For a phase-coherent model, a 1 mHz error
   integrates to a full cycle of phase slip in only ≈ 1000 s; over a 3 h window the
   coherent model and the data decorrelate completely.
2. **The oscillation never stops.** Both records settle at a driven equilibrium amplitude
   of ≈ 16–18 cycles with slow wandering, deep nulls, and revivals — a narrowband
   resonator response to ambient drive, i.e. *signal at exactly the analysis frequency*,
   not measurement noise. (Per-segment residual RMS, the actual broadband noise, is only
   ≈ 1.3–1.9 cycles.)
3. **The decay is not a single exponential** — shown next.

### 3.1 Is there a second mode? (No.)

Beating between closely spaced modes could also produce envelope wander and nulls, so check
the high-resolution spectrum of the early (decay) and late (plateau) portions of each record.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for j, (label, ch) in enumerate({"Pre-Vibe": PRE_CH, "Post-Vibe": POST_CH}.items()):
    rec = records[label]
    t, y, fs = rec["t"], rec["channels"][ch], rec["fs"]
    late_lim = (15000, 28000) if label == "Pre-Vibe" else (20000, 60000)
    for k, (seg_label, mask) in enumerate(
        [("early (decay) 0–8000 s", (t >= 0) & (t < 8000)),
         (f"late (plateau) {late_lim[0]}–{late_lim[1]} s", (t >= late_lim[0]) & (t < late_lim[1]))]
    ):
        seg = y[mask] - np.mean(y[mask])
        x = np.linspace(-1, 1, len(seg))
        seg = seg - np.polyval(np.polyfit(x, seg, 3), x)
        spec = np.abs(np.fft.rfft(seg * np.hanning(len(seg)))) ** 2
        fr = np.fft.rfftfreq(len(seg), 1 / fs)
        band = (fr > 7.6) & (fr < 7.75)
        ax = axes[k, j]
        ax.semilogy(fr[band], spec[band], lw=0.7)
        ax.set_title(f"{label}: {seg_label}")
        ax.set_xlabel("f (Hz)")
        ax.grid(alpha=0.3)
plt.tight_layout()
savefig(fig, "fig04_spectra.png")
plt.show()

A **single dominant mode** in every segment: the nearest secondary features are ≈ 5 orders
of magnitude weaker and consistent with spectral leakage sidebands of the slowly drifting
tone. Multi-mode beating is **ruled out** as the cause of the envelope structure — the
plateau and its nulls/revivals are the amplitude dynamics of one ambient-driven mode.

### 3.2 Amplitude-dependent damping and frequency pull

Fit the local decay rate in amplitude bands and plot the frequency against amplitude.

In [ ]:
AMP_BANDS = [(400, 700), (200, 400), (100, 200), (60, 100)]  # cycles

band_rows = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for label, g in gt.items():
    t_mid, a_seg, f_seg, m = g["t_mid"], g["A"], g["f"], g["decay_mask"]
    floor = g["floor"]
    a_c = np.sqrt(np.maximum(a_seg**2 - floor**2, 1e-12))
    color = "C0" if label == "Pre-Vibe" else "C1"

    # local tau per amplitude band
    for lo, hi in AMP_BANDS:
        mb = m & (a_seg >= lo) & (a_seg < hi)
        if np.sum(mb) < 4:
            continue
        slope, _ = np.polyfit(t_mid[mb], np.log(a_c[mb]), 1)
        tau_b = -1.0 / slope
        q_b = np.pi * np.mean(f_seg[mb]) * tau_b
        band_rows.append(
            {"epoch": label, "amplitude_band_cycles": f"{lo}-{hi}",
             "A_mid": np.sqrt(lo * hi), "n_segments": int(np.sum(mb)),
             "tau_s": tau_b, "Q": q_b}
        )

    # log-envelope residual (shows curvature = non-exponential decay)
    slope, intercept = np.polyfit(t_mid[m], np.log(a_c[m]), 1)
    axes[0].plot(t_mid[m], np.log(a_c[m]) - (slope * t_mid[m] + intercept),
                 ".", ms=4, color=color, label=label)
    # f vs A
    axes[2].semilogx(a_seg[m], f_seg[m], ".", ms=4, color=color, label=label)

band_df = pd.DataFrame(band_rows)
for label in gt:
    sub = band_df[band_df.epoch == label]
    color = "C0" if label == "Pre-Vibe" else "C1"
    axes[1].semilogx(sub.A_mid, sub.Q, "o-", color=color, label=label)

axes[0].set_title("Log-envelope residual of single-exponential fit")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("log residual")
axes[1].set_title("Local Q vs amplitude")
axes[1].set_xlabel("Amplitude (cycles)"); axes[1].set_ylabel("Q")
axes[2].set_title("Frequency vs amplitude")
axes[2].set_xlabel("Amplitude (cycles)"); axes[2].set_ylabel("f (Hz)")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
savefig(fig, "fig05_q_vs_amplitude.png")
plt.show()

band_df.round({"A_mid": 0, "tau_s": 0, "Q": -2})

**The resonator is measurably nonlinear over this amplitude range:**

- **Damping decreases as amplitude decreases** — local Q rises from ≈ 6.5–8.3×10⁴ at high
  amplitude (400–700 cycles) to ≈ 1.2–1.8×10⁵ at low amplitude (100–200 and 60–100 cycle
  bands; the lowest band carries the largest floor-correction uncertainty). The log-envelope residual
  shows smooth, systematic curvature far larger than segment scatter. There is therefore
  **no single "true Q"** for these records: any single-exponential fit returns an average
  that depends on the amplitude range (i.e. the time window) it covers — this is the
  fundamental reason Q estimates here are window-dependent no matter which estimator is used.
- **Frequency decreases with amplitude** approximately linearly in $\log A$ over the
  observed range. The pull is ≈ 5× stronger Pre-Vibe than Post-Vibe — a real physical
  change across the vibration test, quantified in section 4.

## 4. Pre-Vibe vs Post-Vibe comparison (R1)

Because both $f$ and $Q$ depend on amplitude, a defensible comparison must be made **at
matched amplitude**. We compare: frequency evaluated at high amplitude and at the
low-amplitude end of the decay; the frequency–amplitude pull coefficient; Q per amplitude
band; the whole-decay average Q; and the driven plateau level.

In [ ]:
comp = {}
for label, g in gt.items():
    t_mid, a_seg, f_seg, m = g["t_mid"], g["A"], g["f"], g["decay_mask"]
    # frequency vs log-amplitude linear model over the decay region
    coeff = np.polyfit(np.log(a_seg[m]), f_seg[m], 1)
    f_at = lambda A: float(np.polyval(coeff, np.log(A)))
    comp[label] = {
        "f @ A=500 cycles (Hz)": f_at(500.0),
        "f @ A=60 cycles (Hz)": f_at(60.0),
        "freq pull df/dlnA (mHz per e-fold)": 1e3 * coeff[0],
        "whole-decay tau (s)": g["tau"],
        "whole-decay Q": g["Q"],
        "Q @ 400-700 cycles": float(band_df.query("epoch == @label and amplitude_band_cycles == '400-700'").Q.iloc[0]),
        "Q @ 100-200 cycles": float(band_df.query("epoch == @label and amplitude_band_cycles == '100-200'").Q.iloc[0]),
        "Q @ 60-100 cycles": float(band_df.query("epoch == @label and amplitude_band_cycles == '60-100'").Q.iloc[0]),
        "driven plateau (cycles)": g["floor"],
    }

comp_df = pd.DataFrame(comp)
comp_df["Change (Post - Pre)"] = comp_df["Post-Vibe"] - comp_df["Pre-Vibe"]
comp_df["Change (%)"] = 100.0 * (comp_df["Post-Vibe"] - comp_df["Pre-Vibe"]) / comp_df["Pre-Vibe"]

for A_ref in (500.0, 60.0):
    f_pre = comp["Pre-Vibe"][f"f @ A={A_ref:.0f} cycles (Hz)"]
    f_post = comp["Post-Vibe"][f"f @ A={A_ref:.0f} cycles (Hz)"]
    print(f"Frequency shift @ A={A_ref:.0f} cycles: {1e3 * (f_post - f_pre):+.3f} mHz "
          f"({1e6 * (f_post - f_pre) / f_pre:+.1f} ppm)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
x = np.arange(len(AMP_BANDS))
for label, color in [("Pre-Vibe", "C0"), ("Post-Vibe", "C1")]:
    sub = band_df[band_df.epoch == label]
    axes[0].bar(x + (0 if label == "Pre-Vibe" else 0.38), sub.Q, width=0.36,
                color=color, alpha=0.85, label=label)
axes[0].set_xticks(x + 0.19)
axes[0].set_xticklabels([f"{lo}-{hi}" for lo, hi in AMP_BANDS])
axes[0].set_xlabel("Amplitude band (cycles)")
axes[0].set_ylabel("Q")
axes[0].set_title("Q per amplitude band")
axes[0].legend()

for label, color in [("Pre-Vibe", "C0"), ("Post-Vibe", "C1")]:
    g = gt[label]
    m = g["decay_mask"]
    axes[1].semilogx(g["A"][m], g["f"][m], ".", ms=4, color=color, label=label)
axes[1].set_xlabel("Amplitude (cycles)")
axes[1].set_ylabel("f (Hz)")
axes[1].set_title("Frequency vs amplitude: Pre vs Post")
for ax in axes:
    ax.grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
savefig(fig, "fig06_pre_post_comparison.png")
plt.show()

comp_df.round(6)

### 4.1 Comparison summary

- **Frequency:** the resonance shifted **down** across the vibration test, but the shift is
  amplitude-dependent because the anharmonic pull changed: ≈ −0.51 mHz (−66 ppm) at high
  amplitude (A = 500 cycles) vs ≈ −1.21 mHz (−157 ppm) at low amplitude (A = 60 cycles).
  A single-number "frequency shift" from a whole-record coherent fit (the earlier
  notebooks reported −91 ppm) is an average over the decay and lands between these; the
  amplitude-resolved values above are the physically meaningful ones.
- **Frequency pull:** the $\mathrm{d}f/\mathrm{d}\ln A$ coefficient weakened from
  ≈ −0.42 to ≈ −0.09 mHz per amplitude e-fold — a factor ≈ 5. This is a clear,
  well-resolved physical change in the resonator (or its mount/readout chain).
- **Q:** at matched amplitudes the Post-Vibe Q is **equal or higher** than Pre-Vibe:
  whole-decay average **+13 %**, band-resolved +28 % (400–700 cycles), +51 % (100–200),
  +4 % (60–100; this lowest band has few segments and a large floor correction, so treat
  it as consistent-within-uncertainty). There is **no evidence of damage-induced Q
  degradation**. The earlier estimator-dependent claims (−30 % from `Q_profile`, +27 %
  from `Q_nls_raw`) are artifacts, as Part II demonstrates.
- **Both records share** an ambient-driven equilibrium amplitude of ≈ 16–18 cycles and
  baseline wander of tens of cycles; the Post-Vibe record additionally shows strong
  re-excitation episodes late in the record (e.g. near +48 000 s), so any analysis window
  extending past ≈ 15 000 s samples driven dynamics rather than free decay.

**Anomalous/problematic measurements identified:** (i) all non-ring-down channels contain
only crosstalk and must not be fit; (ii) windows that include the plateau or the Post-Vibe
re-excitations bias envelope-slope estimates; (iii) windows offset deep into the record
(e.g. the offset-9000 s specimen below) are dominated by driven dynamics and are unusable
for free-decay Q.

---

## 5. Part II — Why the library's Q estimation fails on these data

Sections 1–3 established what the real signal looks like. Compare that against what each
estimator assumes:

| Assumption of the library model | Reality in the EDU records | Violated? |
|---|---|---|
| Single decaying sinusoid, one mode | Single mode confirmed (§3.1) | No |
| Constant frequency over the fit window | Drifts by 0.3–1.1 mHz during the decay (§3) | **Yes — fatal for coherent fits** |
| Constant phase (global coherence) | Phase slips by many cycles w.r.t. any fixed $f$ | **Yes** |
| Constant damping rate (single $\tau$) | Local $\tau$ varies ≈ 2–3× from the highest to the lowest amplitude band (§3.2) | **Yes — makes "Q" window-dependent** |
| Signal decays to zero; residual is white noise | Settles at a *driven* tone of ≈ 17 cycles at the same frequency, with revivals | **Yes — biases envelope fits** |
| Stationary, zero-mean Gaussian noise | Broadband noise ≈ 1.5 cycles is fine, but the dominant "residual" is the coherent plateau + baseline wander | **Yes** |
| Adequate SNR | SNR ≈ 400 at release — noise is *not* the problem | No |
| Correct sampling interval | Uniform to ≈ 10⁻⁴ relative | No |
| Ring-down fully inside the window | Both records start at the release | No (for offset-0 windows) |

The two decisive violations are **frequency drift** (breaks the phase-coherent NLS/DFT/profile
estimators) and **non-exponential decay + driven plateau** (makes any single-$\tau$ number
window-dependent and biases envelope fits high for long windows). The rest of Part II
demonstrates each mechanism in isolation.

### 5.1 Failure specimen: a window offset deep into the record

What happens if the analysis window misses the main decay (here: Post-Vibe, starting
9000 s after the release — precisely what a unit mix-up of `start_time` produces)?

In [ ]:
# Two numerically equivalent ways to select the same window: (A) reload via
# mokutools with start_time=9000 (offset), (B) mask the already-loaded record.
# The two time arrays agree to float round-off (same N, same span).
pm_a = MokuPhasemeterObject(filename=str(POST_FILE), start_time=9000, duration=3600 * 3.5)
t_a = pm_a.df["time"].values - pm_a.df["time"].values[0]
y_a = pm_a.df["2_cycles"].values
t_b, y_b = window("Post-Vibe", POST_CH, 9000.0, 3600 * 3.5)
print(f"variant A: N={len(t_a)}, span={t_a[-1]:.6f} s | variant B: N={len(t_b)}, span={t_b[-1]:.6f} s")

specimens = {}
for name, (t_w, y_w) in {"A (offset reload)": (t_a, y_a), "B (masked record)": (t_b, y_b)}.items():
    specimens[name] = analyzer.analyze_array(t=t_w, data=y_w, detrend="constant")

spec_df = pd.DataFrame(
    {
        name: {
            "tau_seed_s": r["tau_seed"],
            "tau_seed_method": r["tau_seed_method"],
            "tau_est_s": r["tau_est"],
            "T_crop_s": r["T_crop"],
            "Q_profile": r["Q_profile"],
            "Q_profile_status": r["Q_profile_status"],
            "Q_nls_raw": r["Q_nls_raw"],
            "Q_nls_status": r["Q_nls_status"],
            "Q_envelope": r["Q_envelope"],
        }
        for name, r in specimens.items()
    }
)
display(spec_df)
specimen = specimens["B (masked record)"]

fig, ax = plt.subplots(figsize=(13, 4.5))
plot_q_envelope_overlay(ax, specimen)
ax.set_title("Failure specimen: Post-Vibe window offset +9000 s (plateau-dominated)")
plt.tight_layout()
savefig(fig, "fig07_specimen_offset9000.png")
plt.show()

**Anatomy of the collapse.** In this window the amplitude decays only weakly and the
phase is decoherent w.r.t. any constant frequency. For a *phase-coherent* least-squares
model, a long-$\tau$ constant-frequency sinusoid inevitably drifts **anti-phase** with the
data and *increases* the residual beyond that of no model at all — so the optimizer
prefers to shrink the model away: `tau_est` collapses from the ≈ 5700 s envelope seed to
a small value, and the pipeline then crops the record to $3\,\tau_{\rm est}$, silently
discarding most of the selected data. This is a **cascade design flaw**: a coherence
failure in `estimate_tau` redefines the analysis window with no guard against the
envelope seed it just ignored. (*Fixed 2026-08-19:* the pipeline now cross-checks
`tau_est` against the pre-crop envelope tau and refuses the collapsed crop; re-running
this cell against the current library will no longer reproduce the collapse.)

The two variants above make the pathology vivid: their inputs are identical except for
**float round-off in the time array** (same $N$, same span), yet variant A collapses to
$\tau_{\rm est} \approx 36$ s (crop = 107 s, 0.8 % of the data,
`Q_profile` → `upper_limit`) while variant B lands at $\tau_{\rm est} \approx 1051$ s
(crop = 3154 s, `Q_profile` ≈ 2.8×10⁴ reported **`valid`** — vs the envelope's ≈ 1.3×10⁵).
On decoherent data the objective has no meaningful optimum, so ULP-level input
differences select different local minima — a definitive sign that the *model*, not the
optimizer, is at fault. Neither variant produces a trustworthy result, and only one of
them admits it.

### 5.2 Controlled experiments: reproduce the failure one ingredient at a time

Start from a synthetic record the estimators handle perfectly and add the *measured*
EDU pathologies one at a time. All records: $f_s = 149.012$ Hz, $f_0 = 7.6699$ Hz,
$\tau = 3700$ s ($Q_{\rm true} = \pi f_0 \tau \approx 8.92\times10^4$), $A_0 = 600$ cycles,
white noise $\sigma = 1.5$ cycles, duration 3 h — all matched to the Pre-Vibe record.

| Case | Added ingredient (measured on EDU data) |
|---|---|
| E1 | none (idealized model) |
| E2 | amplitude-proportional frequency pull, $\kappa = -2.06\,\mu$Hz/cycle |
| E2b | linear-in-time frequency drift, $\dot f = +1.08\times10^{-7}$ Hz/s |
| E2c | **the actual measured Pre-Vibe frequency trajectory** $f(t)$, interpolated — run in 4 variants (2 noise seeds × 2 `tau_init` choices) to probe initialization dependence |
| E3 | ambient-driven plateau: AR(2) resonator (same $f_0,\tau$) driven by white noise, equilibrium ≈ 16.5 cycles |
| E4 | E2 + E3 + baseline wander ("EDU twin") |
| E5 | amplitude-dependent damping, $1/\tau(A) = 1/5200\,\mathrm{s} + 2.17\times10^{-7}A$ (no freq drift) |

Each case is analyzed with the full `RingDownAnalyzer` and with the segmented-demodulation
reference estimator (floor-corrected where a plateau is present).

In [ ]:
rng = np.random.default_rng(RNG_SEED)
FS_SYN = 149.012
T_SYN = 10800.0
N_SYN = int(T_SYN * FS_SYN)
t_syn = np.arange(N_SYN) / FS_SYN
F0_SYN, TAU_SYN, A0_SYN, SIGMA_W = 7.6699, 3700.0, 600.0, 1.5
Q_TRUE = np.pi * F0_SYN * TAU_SYN


def driven_plateau(n, fs, f0, tau, rms_target, rng):
    """AR(2) resonator (same f0, tau) driven by white noise -> narrowband plateau."""
    dt = 1.0 / fs
    w0 = 2 * np.pi * f0
    a = 2 - dt * 2 / tau - (w0 * dt) ** 2
    b = -(1 - dt * 2 / tau)
    x = np.zeros(n)
    force = rng.normal(0, 1, n)
    for i in range(2, n):
        x[i] = a * x[i - 1] + b * x[i - 2] + force[i] * dt * dt
    return x * rms_target / np.std(x[n // 2:])


def make_signal(rng, freq_pull=0.0, lin_drift=0.0, f_traj=None, plateau_rms=0.0,
                baseline=False, tau_amp_dep=None):
    if tau_amp_dep is None:
        amp = A0_SYN * np.exp(-t_syn / TAU_SYN)
    else:
        tau_slow, beta = tau_amp_dep
        coarse = 100
        a_c, a_list = A0_SYN, [A0_SYN]
        for _ in range(N_SYN // coarse + 1):
            a_c = a_c * np.exp(-coarse / FS_SYN * (1 / tau_slow + beta * a_c))
            a_list.append(a_c)
        amp = np.interp(t_syn, np.arange(len(a_list)) * coarse / FS_SYN, a_list)
    if f_traj is not None:
        f_inst = f_traj
    else:
        f_inst = F0_SYN + freq_pull * amp + lin_drift * t_syn
    phase = 2 * np.pi * np.cumsum(f_inst) / FS_SYN
    x = amp * np.cos(phase) + rng.normal(0, SIGMA_W, N_SYN)
    if plateau_rms > 0:
        x = x + driven_plateau(N_SYN, FS_SYN, F0_SYN, TAU_SYN, plateau_rms, rng)
    if baseline:
        wander = np.cumsum(rng.normal(0, 1, N_SYN))
        k = int(60 * FS_SYN)
        wander = np.convolve(wander / np.std(wander), np.ones(k) / k, mode="same") * 10.0
        x = x + wander
    return x


def demod_q(t, y, fs, floor=0.0):
    seg = demodulate_segments(t, y, fs)
    t_mid, f_seg, a_seg, _ = seg.T
    if floor > 0:
        m, tau, q, _ = decay_region_fit(t_mid, a_seg, f_seg, floor)
        return q
    m = a_seg > 0.05 * np.max(a_seg)
    slope, _ = np.polyfit(t_mid[m], np.log(a_seg[m]), 1)
    return np.pi * np.mean(f_seg[m]) * (-1.0 / slope)


# Measured Pre-Vibe frequency trajectory for E2c
g_pre = gt["Pre-Vibe"]
m_pre = g_pre["decay_mask"]
f_traj_pre = np.interp(t_syn, g_pre["t_mid"][m_pre], g_pre["f"][m_pre])

exp_cases = {
    "E1 idealized": (dict(), RNG_SEED, {}),
    "E2 amp-prop pull": (dict(freq_pull=-2.06e-6), RNG_SEED, {}),
    "E2b linear drift": (dict(lin_drift=1.077e-7), RNG_SEED, {}),
    "E2c measured f(t)": (dict(f_traj=f_traj_pre), RNG_SEED, {}),
    "E2c + tau_init=2000": (dict(f_traj=f_traj_pre), RNG_SEED, {"tau_init": 2000.0}),
    "E2c seed B": (dict(f_traj=f_traj_pre), 7, {}),
    "E2c seed B + tau_init=2000": (dict(f_traj=f_traj_pre), 7, {"tau_init": 2000.0}),
    "E3 driven plateau": (dict(plateau_rms=16.5 / np.sqrt(2)), RNG_SEED, {}),
    "E4 EDU twin": (dict(freq_pull=-2.06e-6, plateau_rms=16.5 / np.sqrt(2), baseline=True), RNG_SEED, {}),
    "E5 amp-dep damping": (dict(tau_amp_dep=(5200.0, 2.17e-7)), RNG_SEED, {}),
}

exp_rows = []
for name, (kw, seed, an_kwargs) in exp_cases.items():
    y_syn = make_signal(np.random.default_rng(seed), **kw)
    r = analyzer.analyze_array(t=t_syn, data=y_syn, **an_kwargs)
    floor = 16.5 if kw.get("plateau_rms") else 0.0
    q_demod = demod_q(t_syn, y_syn, FS_SYN, floor=floor)
    exp_rows.append(
        {
            "case": name,
            "Q_profile": r["Q_profile"],
            "Q_profile/Q_true": (r["Q_profile"] or np.nan) / Q_TRUE,
            "Q_profile_status": r["Q_profile_status"],
            "Q_nls_raw": r["Q_nls_raw"],
            "Q_envelope": r["Q_envelope"],
            "Q_envelope/Q_true": (r["Q_envelope"] or np.nan) / Q_TRUE,
            "Q_demod": q_demod,
            "Q_demod/Q_true": q_demod / Q_TRUE,
            "tau_est": r["tau_est"],
        }
    )
    print(f"{name}: done")
exp_df = pd.DataFrame(exp_rows).set_index("case")
exp_df.round(3)

**Controlled-experiment results** (values from the table above; `Q_true = 8.92e4` for E1–E4):

- **E1 (idealized):** every estimator is essentially exact — including `Q_profile` to
  4 significant figures. The estimators are *not* algorithmically broken on their own model.
- **E2/E2b (frequency drift only):** `Q_profile` and `Q_nls_raw` are biased by ≈ **+90 %
  and +60 %** respectively while still reporting `valid`; the incoherent envelope and
  demodulation estimators remain within ≈ 0.2 % of truth. Frequency drift alone — at the
  measured magnitude — breaks the coherent estimators.
- **E2c (the actual measured Pre-Vibe $f(t)$, everything else ideal):** the four
  variants (two noise seeds × two initializations, identical drift and decay) return
  `Q_profile` at **0.51×, 1.77×, 1.77×, and 3.80× of truth — every one of them
  `valid`** with a per-mille confidence interval. Under decoherence the least-squares
  landscape has many near-degenerate minima, so the reported Q becomes a lottery over
  inputs that should be irrelevant (here the initialization matters more than the seed).
  This is exactly the real-data phenomenology: the canonical windows in §2 returned
  0.65× (Pre) and 0.25× (Post) of the drift-immune reference — with the *same* Pre
  window returning 0.45× in an earlier session differing only at round-off level — and
  the window sweep in §5.4 scatters by 12×. The incoherent estimators are unaffected
  (≤ 0.2 % across all four variants).
- **E3 (plateau only):** the coherent estimators are unaffected (the plateau is
  random-phase and averages out of a coherent fit); the envelope estimator picks up a
  small positive bias in a 3 h window that grows with window length (quantified in §5.3).
- **E4 (EDU twin):** reproduces the full real-data pattern — confidently wrong coherent Q,
  mildly biased envelope Q, accurate demodulation reference.
- **E5 (amplitude-dependent damping):** every single-$\tau$ estimator returns a different
  window-average (profile ≈ 9.6×10⁴, envelope/demod ≈ 1.04×10⁵ here); none is "wrong",
  but no single number characterizes the decay — matching §3.2.

### 5.3 Sensitivity scans: quantitative failure thresholds

**(a) Coherent profile Q vs frequency error.** Fix the model frequency at
$f_0 + \delta\!f$ on the idealized record E1 and profile $\tau$:

In [ ]:
y_ideal = make_signal(np.random.default_rng(RNG_SEED))
profiler = ProfileQEstimator()

df_scan = np.array([0.0, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4])
scan_rows = []
for df_err in df_scan:
    pr = profiler.estimate(t_syn, y_ideal, FS_SYN, f_init=F0_SYN + df_err, tau_init=TAU_SYN)
    scan_rows.append(
        {"df_Hz": df_err, "df_times_tau": df_err * TAU_SYN,
         "tau_hat": pr.tau_hat, "Q": pr.Q, "Q/Q_true": (pr.Q or np.nan) / Q_TRUE,
         "status": pr.status}
    )
scan_df = pd.DataFrame(scan_rows)
display(scan_df.round(4))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogx(scan_df.df_times_tau[1:], scan_df["Q/Q_true"][1:], "o-")
ax.axhline(1.0, color="k", ls=":", lw=1)
ax.axhline(0.95, color="r", ls="--", lw=1, label="5 % bias")
ax.set_xlabel(r"$\delta\!f \cdot \tau$  (frequency error × decay time)")
ax.set_ylabel(r"$Q_{\rm profile} / Q_{\rm true}$")
ax.set_title("Profile-Q bias vs fixed-frequency error (all points report status='valid')")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
savefig(fig, "fig08_df_tau_scan.png")
plt.show()

The bias follows the dimensionless product $\delta\!f\cdot\tau$: it stays below ≈ 2 % only
for $\delta\!f\cdot\tau \lesssim 0.01$ — i.e. the fixed model frequency must be correct to
**≈ 3 µHz** for $\tau \approx 3700$ s. The EDU resonator's own frequency *moves* by
300–1100 µHz during the decay, exceeding the tolerance by 2–3 orders of magnitude. Every
biased point still reports `status='valid'` with a tight CI: **the profile-likelihood
validity machinery cannot detect model misspecification**, because it only measures RSS
curvature against the (misspecified) white-noise model.

**(b) Envelope Q vs window length in the presence of the driven plateau.** The
`q_envelope_diagnostic` amplitude floor (5 % of max) does not model the plateau; late
windows sit at the plateau level and flatten the fitted slope:

In [ ]:
# Synthetic: 6 h record with plateau, evaluate envelope Q on nested windows
rng2 = np.random.default_rng(RNG_SEED + 1)
T6 = 6 * 3600.0
N6 = int(T6 * FS_SYN)
t6 = np.arange(N6) / FS_SYN
y6 = (A0_SYN * np.exp(-t6 / TAU_SYN) * np.cos(2 * np.pi * F0_SYN * t6)
      + rng2.normal(0, SIGMA_W, N6)
      + driven_plateau(N6, FS_SYN, F0_SYN, TAU_SYN, 16.5 / np.sqrt(2), rng2))

hours_list = [1, 2, 3, 4, 6]
syn_env, real_env = [], {"Pre-Vibe": [], "Post-Vibe": []}
for hours in hours_list:
    n = int(hours * 3600 * FS_SYN)
    d = q_envelope_diagnostic(t6[:n], y6[:n], F0_SYN)
    syn_env.append(d.Q / Q_TRUE)
    for label, ch in {"Pre-Vibe": PRE_CH, "Post-Vibe": POST_CH}.items():
        t_w, y_w = window(label, ch, 0.0, hours * 3600.0)
        d = q_envelope_diagnostic(t_w, y_w, F_NOMINAL)
        real_env[label].append(d.Q)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(hours_list, syn_env, "o-")
axes[0].axhline(1.0, color="k", ls=":")
axes[0].set_xlabel("Window length (h)")
axes[0].set_ylabel(r"$Q_{\rm envelope}/Q_{\rm true}$")
axes[0].set_title("Synthetic (plateau 16.5 cycles): envelope-Q bias vs window")
for label, color in [("Pre-Vibe", "C0"), ("Post-Vibe", "C1")]:
    axes[1].plot(hours_list, real_env[label], "o-", color=color, label=label)
    axes[1].axhline(gt[label]["Q"], color=color, ls=":", lw=1,
                    label=f"{label} demod reference")
axes[1].set_xlabel("Window length (h)")
axes[1].set_ylabel("Q_envelope")
axes[1].set_title("Real data: envelope Q vs window length")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
savefig(fig, "fig09_envelope_window_bias.png")
plt.show()

On the synthetic record the envelope-Q bias grows from ≈ 0 (1–2 h) to ≈ +7 % (6 h) as
plateau windows enter the fit. On the real data the growth is steeper (+30–80 % from 1 h to
6 h) because the plateau bias **adds to the genuine decay-rate curvature** of §3.2 (later
windows sample intrinsically slower decay). The envelope estimator is therefore
*approximately right* for windows that end near where the decay meets ≈ 3× the plateau
level, and increasingly biased high beyond that.

### 5.4 Window sensitivity on the real data

Finally, sweep start offset and duration on both records and compare the three library
estimators against the amplitude-matched demodulation reference. A trustworthy estimator
should be stable across reasonable windows.

In [ ]:
sens_rows = []
for label, ch, kw in [("Pre-Vibe", PRE_CH, {}), ("Post-Vibe", POST_CH, {"detrend": "constant"})]:
    for start in (0.0, 1800.0):
        for hours in (1, 2, 3, 4, 6):
            t_w, y_w = window(label, ch, start, hours * 3600.0)
            r = analyzer.analyze_array(t=t_w, data=y_w, **kw)
            sens_rows.append(
                {"epoch": label, "start_s": start, "duration_h": hours,
                 "Q_profile": r["Q_profile"], "Q_profile_status": r["Q_profile_status"],
                 "Q_nls_raw": r["Q_nls_raw"], "Q_envelope": r["Q_envelope"]}
            )
sens_df = pd.DataFrame(sens_rows)
for col in ("Q_profile", "Q_nls_raw", "Q_envelope"):
    sens_df[col] = pd.to_numeric(sens_df[col], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, label in zip(axes, ["Pre-Vibe", "Post-Vibe"]):
    sub = sens_df[sens_df.epoch == label]
    xj = sub.duration_h + np.where(sub.start_s > 0, 0.12, -0.12)
    ax.semilogy(xj, sub.Q_profile, "o", color="C3", label="Q_profile")
    ax.semilogy(xj, sub.Q_nls_raw, "s", color="C4", label="Q_nls_raw")
    ax.semilogy(xj, sub.Q_envelope, "^", color="C0", label="Q_envelope")
    g = gt[label]
    ax.axhline(g["Q"], color="k", lw=1.2, label="demod reference (whole decay)")
    band = band_df[band_df.epoch == label]
    ax.axhspan(band.Q.min(), band.Q.max(), color="k", alpha=0.10,
               label="physical Q(A) range")
    ax.set_xlabel("Window duration (h)  [two start offsets each]")
    ax.set_title(label)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Q estimate")
axes[0].legend(fontsize=8, loc="lower right")
plt.tight_layout()
savefig(fig, "fig10_window_sensitivity.png")
plt.show()

summary_stats = sens_df.groupby("epoch")[["Q_profile", "Q_nls_raw", "Q_envelope"]].agg(
    ["min", "max", "median"]
)
summary_stats["Q_profile", "max/min"] = (
    summary_stats["Q_profile", "max"] / summary_stats["Q_profile", "min"]
)
summary_stats.round(0)

**Window-sensitivity findings:**

- `Q_profile` scatters by up to an **order of magnitude** across ordinary window choices on
  the Pre-Vibe record (≈ 6.5×10³ … 7.5×10⁴) — while reporting `valid` with per-mille CIs on
  most windows. On Post-Vibe it is *stably wrong* (≈ 2.7×10⁴, i.e. ≈ 3–4× below the
  physical range) because the Post-Vibe drift trajectory is smoother.
- `Q_nls_raw` scatters by ≈ 4× (it re-fits the frequency per window, so its decoherence
  compromise changes with the window).
- `Q_envelope` is the only library estimator that stays inside or near the physical
  Q(A) band, with a systematic upward drift for longer windows (plateau + real curvature).
- The demodulation reference (and its per-amplitude-band values) is stable by construction
  and consistent between both records.

## 6. Conclusions

**Pre-Vibe vs Post-Vibe (Goal 1):**

1. Only resonator R1 rang down on both days; all other channels contain crosstalk only.
2. The resonance frequency shifted down by ≈ 66 ppm (measured at high amplitude) to
   ≈ 157 ppm (low amplitude); the amplitude–frequency pull coefficient weakened ≈ 5×.
3. At matched amplitude, Q did **not** degrade across the vibration test — whole-decay
   average +13 %, band-resolved +4 % to +51 % (all non-negative). Earlier claims of a
   −30 % Q change were estimator artifacts.
4. Both records show amplitude-dependent damping (local Q ≈ 6.5×10⁴ → 1.8×10⁵ over the
   decay), an ambient-driven plateau of ≈ 17 cycles, and baseline wander.

**Q-estimation failure (Goal 2)** — demonstrated mechanisms, in order of importance:

1. **Frequency drift breaks every phase-coherent estimator** (`Q_profile`, `Q_nls`,
   `Q_dft`). The measured 0.3–1.1 mHz drift exceeds the ≈ 3 µHz coherence tolerance
   ($\delta\!f\cdot\tau \lesssim 0.01$) by orders of magnitude. Injecting the measured
   $f(t)$ into an otherwise ideal synthetic record (E2c) makes `Q_profile` a lottery over
   noise seed and initialization — anywhere from 0.51× to 3.80× of truth, always `valid`
   with per-mille CIs — reproducing the real-data phenomenology (canonical windows:
   0.65× Pre, 0.25× Post; 12× scatter across window choices). The `valid`/CI machinery
   cannot detect it.
2. **Coherence collapse of `tau_est` silently destroys the pipeline window** (offset
   specimen in §5.1): the full-record NLS shrinks $\tau$ to fit only a coherent early
   fraction, and the $3\tau$ crop then discards the data.
3. **The decay is genuinely non-exponential** (amplitude-dependent damping), so *any*
   single-$\tau$ estimator is window-dependent; this is a model limitation, not an
   estimator bug, and must be handled by reporting Q as a function of amplitude (or
   fitting a nonlinear-damping model).
4. **The driven plateau biases envelope-slope estimators high** for windows extending past
   the point where the decay meets ≈ 3× the plateau level; the effect grows with window
   length and adds to (3).
5. **Broadband noise is *not* the cause** — SNR ≈ 400 at release, and the estimators are
   exact on matched-SNR idealized synthetics (E1).

See `docs/investigations/20260818_q_estimation_failure_investigation.md` for the full
root-cause report, recommendations, and the proposed library-evolution plan.

> **Implementation status (2026-08-19).** The evolution plan was implemented in full:
> guardrails for mechanisms 1, 2 and 4 (drift gate, crop-cascade guard,
> envelope-mismatch gating), the segmented-demodulation estimator of section 3 as
> `ringdownanalysis.demod.SegmentedDemodEstimator` (drift-immune, plateau-corrected,
> amplitude-resolved — addressing mechanisms 3 and 4), a nonlinear-damping / frequency-pull
> model (`ringdownanalysis.nonlinear`), and an estimator-selection layer
> (`ringdownanalysis.selection`, surfaced as `Q_selected`). The real-data numbers above are
> pinned as regression tests in `tests/test_real_data_regression.py`; the current-library
> demonstration lives in `notebooks/20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb`.
> Note the shipped estimator fits the decay with a Theil-Sen (median-of-slopes) line for
> robustness against plateau revivals, so its whole-decay Q lands 5-10 % below this
> notebook's unweighted least-squares reference on these genuinely curved decays
> (Pre 8.13×10⁴ vs 8.92×10⁴, Post 9.54×10⁴ vs 1.005×10⁵) — a recipe difference from
> decay-rate curvature, not an estimator discrepancy.